# 📅 2026-04-29 (수요일) 개발 노트 : FastAPI 추천 서버 세팅 및 DB 적재 트러블슈팅

## 🎯 오늘의 목표
- [x] Django와 FastAPI의 Two-Tower 백엔드 아키텍처 설계 및 파일 구조 세팅
- [x] PostgreSQL DB 접속하여 데이터 정상 적재 여부 쿼리(SQL) 검증
- [ ] FastAPI Swagger UI를 통한 추천 알고리즘 API 테스트 (지연)

## 🛠 진행 상황 및 핵심 기록
- **FastAPI 구조 확립:** 60개 지표를 활용하는 49차원(수치) 추천 엔진 아키텍처 생성 (models, schemas, routers, services 분리 완벽 적용).
- **DB 검증:** Docker를 구동하고 `psql`로 `hidden_gem_db`에 직접 접속. `\dt`, `SELECT COUNT(*)` 등을 통해 4,190개의 데이터 행이 존재함을 확인.

## 🚨 트러블슈팅 (문제 및 해결)
- **문제:** `games` 테이블의 `name`, `genres` 등이 비어있고, `game_metrics`의 `gem_potential` 지표가 `NULL`로 삽입되는 현상 발견.
- **원인:** 데이터 소스인 `final_master_games.jsonl` 파일 자체를 `$ head -1` 명령어로 까본 결과, 병합된 JSON 안에 게임 메타데이터 키(`name`, `genres` 등)가 아예 존재하지 않음. **이전 단계인 병합 스크립트(`merge_metrics.py`) 로직 오류로 원본 데이터가 유실됨.**
- **해결 방안:** (다음 작업) `merge_metrics.py` 코드를 수정하여 원본 스팀 데이터의 기본 정보들을 유지한 채로 AI 결과를 덮어쓰도록(update) 로직을 수정하고 파일을 재병합해야 함.

## 💡 인사이트 및 다음 할 일
- **배운 점:** "DB에 값이 안 들어간다"고 무작정 적재 스크립트(`load_games.py`)나 DB를 의심하기 전에, **입력되는 'Raw 데이터(JSON 파일)' 자체를 눈으로 직접 확인해보는 것**이 시니어의 디버깅 1원칙임을 뼈저리게 체득함.
- **다음 할 일:** 1. `merge_metrics.py` 로직 수정 후 `final_master_games.jsonl` 재성성
  2. `python manage.py load_games --update` 명령어로 DB 다시 채우기
  3. FastAPI 서버 켜고 `http://localhost:8000/docs`에서 추천 API 쏴보기